In [ ]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_log_error
os.chdir("../")
os.getcwd()

'/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting'

In [2]:
sales_df = pd.read_csv('data/sales_df.csv')
sales_df['date'] = pd.to_datetime(sales_df['date'])


In [3]:
sales_df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'store_type', 'cluster', 'dcoilwtico', 'holiday_type',
       'is_holiday_local', 'is_holiday_regional', 'is_holiday_national',
       'is_store_closed', 'day_of_week', 'month', 'year', 'time_idx',
       'is_weekend', 'is_payday', 'month_sin', 'month_cos', 'day_sin',
       'day_cos', 'is_earthquake_impact', 'days_to_christmas', 'sales_lag_1',
       'sales_lag_2', 'sales_lag_3', 'sales_lag_4', 'sales_lag_5',
       'sales_lag_6', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21',
       'sales_lag_28', 'sales_lag_56', 'sales_lag_364', 'rolling_mean_7',
       'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7',
       'store_family_velocity', 'target_enc_store_family', 'oil_trend_30',
       'promo_ratio_vs_avg', 'promo_during_payday'],
      dtype='object')

# 1 - Expanding Window à 8 semaines

In [ ]:
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))


def evaluate_expanding_window(serie, model_func, n_splits=3, test_size=56):
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(serie)):
        train_data, val_data = serie[train_idx], serie[val_idx]
    
        preds = model_func(train_data, len(val_data))
        
        score = rmsle(val_data, preds)
        scores.append(score)
        print(f"  Fold {fold+1}: RMSLE = {score:.4f}")
        
    return np.mean(scores)


# 2 - Modèle ARMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX


def run_arma(train_data, forecast_steps):
    model = SARIMAX(train_data, order=(1, 0, 1), 
                    enforce_stationarity=False, 
                    enforce_invertibility=False)
    results = model.fit(disp=False)
    return results.forecast(steps=forecast_steps)


families = sales_df['family'].unique()
summary_results = {}

print(f"🚀 Lancement de l'ARMA sur {len(families)} familles...")

for fam in families[:5]: # On teste les 5 premières pour commencer
    print(f"\nAnalyse de la famille : {fam}")
    
    # Filtrage par famille et magasin 1
    daily_data = sales_df[(sales_df['store_nbr'] == 1) & (sales_df['family'] == fam)].sort_values('date')
    serie_val = daily_data['sales'].fillna(0).values
    
    if len(serie_val) > 200: # Vérification
        avg_score = evaluate_expanding_window(serie_val, run_arma)
        summary_results[fam] = avg_score
        print(f">> Score Moyen {fam} : {avg_score:.4f}")

print("\n--- Synthèse ARMA ---")
for fam, score in summary_results.items():
    print(f"{fam}: {score:.4f}")

🚀 Lancement de l'ARMA sur 33 familles...

Analyse de la famille : AUTOMOTIVE
  Fold 1: RMSLE = 0.5387
  Fold 2: RMSLE = 0.5842
  Fold 3: RMSLE = 0.6138
>> Score Moyen AUTOMOTIVE : 0.5789

Analyse de la famille : BABY CARE


/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Fold 1: RMSLE = 0.0000
  Fold 2: RMSLE = 0.0000
  Fold 3: RMSLE = 0.0000
>> Score Moyen BABY CARE : 0.0000

Analyse de la famille : BEAUTY
  Fold 1: RMSLE = 0.6348
  Fold 2: RMSLE = 0.5580
  Fold 3: RMSLE = 0.4527
>> Score Moyen BEAUTY : 0.5485

Analyse de la famille : BEVERAGES
  Fold 1: RMSLE = 0.4132
  Fold 2: RMSLE = 0.3500
  Fold 3: RMSLE = 0.3361
>> Score Moyen BEVERAGES : 0.3664

Analyse de la famille : BOOKS
  Fold 1: RMSLE = 0.3906
  Fold 2: RMSLE = 0.4318
  Fold 3: RMSLE = 0.3925
>> Score Moyen BOOKS : 0.4050

--- Synthèse ARMA ---
AUTOMOTIVE: 0.5789
BABY CARE: 0.0000
BEAUTY: 0.5485
BEVERAGES: 0.3664
BOOKS: 0.4050


## 3 - SARIMAX

In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def run_sarimax_exog(train_data, train_exog, forecast_steps, test_exog):
    """
    train_data: les ventes passées
    train_exog: les variables type promotion/payday sur la période de train
    test_exog: les mêmes variables sur la période de prédiction (8 semaines)
    """
    # Configuration (1,1,1)x(1,1,1,7) pour capter tendance + semaine + exogènes
    model = SARIMAX(train_data, 
                    exog=train_exog,
                    order=(1, 1, 1), 
                    seasonal_order=(1, 1, 1, 7),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    
    results = model.fit(disp=False)
    
    # On doit donner les variables exogènes du futur pour prédire le futur
    forecast = results.forecast(steps=forecast_steps, exog=test_exog)
    return np.maximum(0, forecast)

In [ ]:
exog_cols = [
    'onpromotion',       
    'dcoilwtico',        
    'is_payday',         
    'is_holiday_national', 
    'is_weekend',        
    'month_sin', 'month_cos', 
    'day_sin', 'day_cos'      
]
summary_sarimax = {}

for fam in sales_df['family'].unique():
    print(f"\n🚀 Test SARIMAX + Exogènes pour : {fam}")
    
    
    df_fam = sales_df[(sales_df['store_nbr'] == 1) & (sales_df['family'] == fam)].sort_values('date')
    

    serie_y = df_fam['sales'].fillna(0).values
    serie_x = df_fam[exog_cols].fillna(0).values
    

    tscv = TimeSeriesSplit(n_splits=3, test_size=56)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(serie_y)):

        y_train, y_val = serie_y[train_idx], serie_y[val_idx]
        x_train, x_val = serie_x[train_idx], serie_x[val_idx]
        

        preds = run_sarimax_exog(y_train, x_train, len(y_val), x_val)
        
        score = rmsle(y_val, preds)
        scores.append(score)
        print(f"  Fold {fold+1}: RMSLE = {score:.4f}")
    
    summary_sarimax[fam] = np.mean(scores)
    print(f">> Score Moyen SARIMAX {fam} : {summary_sarimax[fam]:.4f}")
print("\n--- Synthèse SARIMAX + Exogènes ---")
for fam, score in summary_sarimax.items():
    print(f"{fam}: {score:.4f}")



🚀 Test SARIMAX + Exogènes pour : AUTOMOTIVE


/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Fold 1: RMSLE = 0.5713


/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Fold 2: RMSLE = 0.5467


...

  Fold 2: RMSLE = 0.4221
  Fold 3: RMSLE = 0.5033
>> Score Moyen SARIMAX BOOKS : 0.4414

🚀 Test SARIMAX + Exogènes pour : BREAD/BAKERY


/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Fold 1: RMSLE = 1.4276


/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Fold 2: RMSLE = 0.2503
